In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

In [2]:
PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent

RAW = PROJECT / "data" / "raw"
PROCESSED = PROJECT / "data" / "processed"

V4_SHORT_PATH = PROCESSED / "cases_v4_short_merged.csv"
V4_FULL_PATH = PROCESSED / "cases_v4_full_merged.csv"
VDEM_PATH = RAW / "VDEMv16SelectedVars_data.csv"

V5_SHORT_PATH = PROCESSED / "cases_v5_short.csv"
V5_FULL_PATH = PROCESSED / "cases_v5_full.csv"

NEW_VAR = "v2mecenefm"
NEW_COLS = [NEW_VAR, f"{NEW_VAR}_lag1", f"{NEW_VAR}_lag2"]

In [3]:
cases_v4_short = pd.read_csv(V4_SHORT_PATH)
cases_v4_full = pd.read_csv(V4_FULL_PATH)
vdem = pd.read_csv(VDEM_PATH)

vdem.columns = vdem.columns.str.strip()

print("v4 short:", cases_v4_short.shape)
print("v4 full:", cases_v4_full.shape)
print("V-Dem:", vdem.shape)

v4 short: (1075, 111)
v4 full: (1075, 247)
V-Dem: (28092, 20)


In [4]:
required_case_cols = {"case_id", "country_clean", "year", "vdem_matched"}
required_vdem_cols = {"country_name", "year", NEW_VAR}

for label, df in {"short": cases_v4_short, "full": cases_v4_full}.items():
    missing = required_case_cols - set(df.columns)
    if missing:
        raise ValueError(f"{label} v4 dataset is missing required columns: {sorted(missing)}")

missing = required_vdem_cols - set(vdem.columns)
if missing:
    raise ValueError(f"V-Dem file is missing required columns: {sorted(missing)}")

for label, df in {"short": cases_v4_short, "full": cases_v4_full}.items():
    existing_new_cols = [col for col in NEW_COLS if col in df.columns]
    print(label, "existing new columns:", existing_new_cols or "none")
    print(label, "duplicate case_id:", df["case_id"].duplicated().sum())

vdem_dupes = vdem.duplicated(["country_name", "year"]).sum()
print("V-Dem duplicate country-years:", vdem_dupes)
if vdem_dupes:
    raise ValueError("V-Dem country-year keys are not unique.")

ValueError: V-Dem file is missing required columns: ['v2mecenefm']

In [ ]:
vdem_press = vdem[["country_name", "year", NEW_VAR]].copy()
vdem_press["country_name"] = vdem_press["country_name"].astype(str).str.strip()
vdem_press["year"] = vdem_press["year"].astype(int)
vdem_press = vdem_press.rename(columns={"country_name": "country_clean"})
vdem_press["_source_present"] = True

vdem_press.head()

In [ ]:
def add_press_freedom(cases: pd.DataFrame, label: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Add v2mecenefm current, lag1, and lag2 while preserving the v4 schema order."""
    out = cases.copy()
    original_columns = list(out.columns)
    original_case_ids = out["case_id"].astype(str).tolist()

    existing_new_cols = [col for col in NEW_COLS if col in out.columns]
    if existing_new_cols:
        out = out.drop(columns=existing_new_cols)
        original_columns = [col for col in original_columns if col not in existing_new_cols]

    out["country_clean"] = out["country_clean"].astype(str).str.strip()
    out["year"] = out["year"].astype(int)

    diagnostics = []

    for lag in [0, 1, 2]:
        col = NEW_VAR if lag == 0 else f"{NEW_VAR}_lag{lag}"
        source_present_col = f"_{col}_source_present"

        temp = vdem_press.copy()
        temp["year"] = temp["year"] + lag
        temp = temp.rename(columns={NEW_VAR: col, "_source_present": source_present_col})

        out = out.merge(
            temp,
            on=["country_clean", "year"],
            how="left",
            validate="many_to_one",
            sort=False,
        )

        diagnostics.append(
            {
                "dataset": label,
                "column": col,
                "source_rows_missing": int(out[source_present_col].isna().sum()),
                "values_missing": int(out[col].isna().sum()),
                "values_nonmissing": int(out[col].notna().sum()),
            }
        )
        out = out.drop(columns=source_present_col)

    insert_at = original_columns.index("vdem_matched")
    final_columns = original_columns[:insert_at] + NEW_COLS + original_columns[insert_at:]
    out = out[final_columns]

    if [col for col in out.columns if col not in NEW_COLS] != original_columns:
        raise AssertionError(f"{label}: existing v4 columns changed order.")
    if out["case_id"].astype(str).tolist() != original_case_ids:
        raise AssertionError(f"{label}: row order or case IDs changed.")
    if len(out) != len(cases):
        raise AssertionError(f"{label}: row count changed.")

    return out, pd.DataFrame(diagnostics)


cases_v5_short, diagnostics_short = add_press_freedom(cases_v4_short, "short")
cases_v5_full, diagnostics_full = add_press_freedom(cases_v4_full, "full")

diagnostics = pd.concat([diagnostics_short, diagnostics_full], ignore_index=True)
diagnostics

In [ ]:
def compare_existing_vdem_lag_logic(cases: pd.DataFrame, label: str, source_var: str = "v2x_polyarchy") -> pd.DataFrame:
    source = vdem[["country_name", "year", source_var]].copy()
    source["country_name"] = source["country_name"].astype(str).str.strip()
    source["year"] = source["year"].astype(int)
    source = source.rename(columns={"country_name": "country_clean"})

    checks = []
    keys = cases[["country_clean", "year"]].copy()
    keys["country_clean"] = keys["country_clean"].astype(str).str.strip()
    keys["year"] = keys["year"].astype(int)

    for lag in [0, 1, 2]:
        existing_col = source_var if lag == 0 else f"{source_var}_lag{lag}"
        expected_col = f"_expected_{existing_col}"

        temp = source.copy()
        temp["year"] = temp["year"] + lag
        temp = temp.rename(columns={source_var: expected_col})

        expected = keys.merge(
            temp,
            on=["country_clean", "year"],
            how="left",
            validate="many_to_one",
            sort=False,
        )[expected_col]

        left = pd.to_numeric(cases[existing_col], errors="coerce").to_numpy()
        right = pd.to_numeric(expected, errors="coerce").to_numpy()
        mismatches = int((~np.isclose(left, right, equal_nan=True)).sum())
        checks.append({"dataset": label, "column": existing_col, "mismatches": mismatches})

    return pd.DataFrame(checks)


lag_logic_check = pd.concat(
    [
        compare_existing_vdem_lag_logic(cases_v5_short, "short"),
        compare_existing_vdem_lag_logic(cases_v5_full, "full"),
    ],
    ignore_index=True,
)

lag_logic_check

In [ ]:
if lag_logic_check["mismatches"].sum() != 0:
    raise AssertionError("The lag merge logic does not match the existing v4 V-Dem lag columns.")

for label, old, new in [
    ("short", cases_v4_short, cases_v5_short),
    ("full", cases_v4_full, cases_v5_full),
]:
    expected_shape = (old.shape[0], old.shape[1] + len(NEW_COLS))
    if new.shape != expected_shape:
        raise AssertionError(f"{label}: expected shape {expected_shape}, got {new.shape}.")
    print(label, "old shape:", old.shape, "new shape:", new.shape)

cases_v5_short.to_csv(V5_SHORT_PATH, index=False)
cases_v5_full.to_csv(V5_FULL_PATH, index=False)

print("Saved:", V5_SHORT_PATH)
print("Saved:", V5_FULL_PATH)

In [ ]:
written_short = pd.read_csv(V5_SHORT_PATH)
written_full = pd.read_csv(V5_FULL_PATH)

summary = pd.DataFrame(
    [
        {
            "dataset": "short",
            "rows": len(written_short),
            "columns": written_short.shape[1],
            "new_columns": [col for col in NEW_COLS if col in written_short.columns],
            "duplicate_case_id": int(written_short["case_id"].duplicated().sum()),
        },
        {
            "dataset": "full",
            "rows": len(written_full),
            "columns": written_full.shape[1],
            "new_columns": [col for col in NEW_COLS if col in written_full.columns],
            "duplicate_case_id": int(written_full["case_id"].duplicated().sum()),
        },
    ]
)

summary